# VoiceShop — train YOLO on GPU

1. Runtime → Change runtime type → **GPU (T4)**
2. Upload `voiceshop_yolo.zip` from Desktop (~23 MB)
3. Run all cells
4. Download `shopsee_yolov8n.pt` → `backend/`

In [ ]:
!pip -q install -U ultralytics
from google.colab import files
print('Upload voiceshop_yolo.zip')
uploaded = files.upload()

In [ ]:
import zipfile, pathlib, shutil

def norm(p: str) -> str:
    return p.replace(chr(92), '/')

zips = sorted(pathlib.Path('.').glob('*.zip'), key=lambda p: p.stat().st_size, reverse=True)
print('found zips:', [(p.name, f'{p.stat().st_size/1e6:.1f} MB') for p in zips])
assert zips, 'upload voiceshop_yolo.zip'
zp = zips[0]
assert zp.stat().st_size > 5_000_000, 'zip too small'

shutil.rmtree('dataset', ignore_errors=True)
root = pathlib.Path('dataset')
root.mkdir()

with zipfile.ZipFile(zp) as z:
    names = [norm(n) for n in z.namelist()]
    n_train = sum(1 for n in names if 'images/train/' in n and not n.endswith('/'))
    print('inside zip train images:', n_train)
    print('sample paths:', names[:5])
    assert n_train > 50, 'zip has no train images'
    for info in z.infolist():
        rel = norm(info.filename)
        target = root / rel
        if info.is_dir() or rel.endswith('/'):
            target.mkdir(parents=True, exist_ok=True)
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        with z.open(info) as src, open(target, 'wb') as dst:
            dst.write(src.read())

yaml = next(root.rglob('data.yaml'))
ds_root = yaml.parent.resolve()
yaml.write_text(
f'''# VoiceShop (Colab)\npath: {ds_root.as_posix()}\ntrain: images/train\nval: images/val\n\nnames:\n  0: apple\n  1: banana\n  2: milk_bottle\n'''
)
train_dir = ds_root / 'images' / 'train'
n = len(list(train_dir.glob('*')))
print('yaml:\n', yaml.read_text())
print('extracted train images:', n)
assert n > 50, 'extract failed'
DATA_YAML = str(yaml)
print('DATA_YAML =', DATA_YAML)

In [ ]:
from ultralytics import YOLO
import pathlib
yaml = next(pathlib.Path('dataset').rglob('data.yaml'))
model = YOLO('yolov8n.pt')
model.train(
    data=str(yaml.resolve()),
    epochs=60,
    imgsz=640,
    batch=16,
    device=0,
    patience=20,
    freeze=10,
    lr0=0.004,
    optimizer='AdamW',
    cos_lr=True,
    mosaic=0.15,
    mixup=0.0,
    close_mosaic=20,
    name='voiceshop',
    exist_ok=True,
)
print('done')

In [ ]:
import shutil, pathlib
from google.colab import files
best = next(pathlib.Path('runs').rglob('best.pt'))
print('best:', best)
shutil.copy(best, 'shopsee_yolov8n.pt')
files.download('shopsee_yolov8n.pt')